# 🌿 Plant Disease Detection using CNN
## Notebook 1 — Dataset & Preprocessing (Phase 1)

**Course:** Artificial Neural Networks & Deep Learning (AI253IA) · **Team:** Swayam Goyal, Yash Bhalotia, Yash Purwar

### What this notebook does
1. Downloads the real **PlantVillage** dataset (54,305 leaf photos, 38 classes, 14 crops).
2. Explores it: how many images per class, what they look like, their size.
3. Splits it into **train / validation / test** sets (70 / 15 / 15 %).
4. Builds the **preprocessing pipeline**: resize → normalise → augment.

### How to run
`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. When asked, allow access to Google Drive.

**Dataset citation:** Hughes, D. P., & Salathé, M. (2015). *An open access repository of images on plant health to enable the development of mobile disease diagnostics.* arXiv:1511.08060.

## Step 0 — Setup
Imports the libraries and creates folders in your Google Drive where results are saved.

In [ ]:
# ============================================================
# SETUP: imports, settings and project folders
# ============================================================
import os, sys, json, time, random, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42                     # fixed "random seed" -> same results every run
random.seed(SEED); np.random.seed(SEED)

IMG_SIZE = 224                # every image is resized to 224 x 224 pixels
BATCH_SIZE = 32               # how many images the model sees at once

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Google Drive keeps our trained models & results safe even if Colab disconnects
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/plant-disease-project'
    DATA_ROOT = '/content/data'              # fast local disk for the images
else:
    PROJECT_DIR = os.path.abspath('../outputs')
    DATA_ROOT = os.path.abspath('../data')

MODELS_DIR  = os.path.join(PROJECT_DIR, 'models')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FIG_DIR     = os.path.join(RESULTS_DIR, 'figures')
for d in [MODELS_DIR, RESULTS_DIR, FIG_DIR, DATA_ROOT]:
    os.makedirs(d, exist_ok=True)

DATASET_DIR = os.path.join(DATA_ROOT, 'PlantVillage-Dataset', 'raw', 'color')
print('Running in Colab:', IN_COLAB)
print('Results will be saved in:', PROJECT_DIR)

## Step 1 — Download the dataset

In [ ]:
# ============================================================
# DOWNLOAD THE PLANTVILLAGE DATASET (about 1 GB, takes ~2-5 minutes)
# Source: the official GitHub copy by S. P. Mohanty (author of our main reference paper)
# We download only the 'raw/color' folder = 54,305 original colour leaf photos.
# ============================================================
REPO_URL = 'https://github.com/spMohanty/PlantVillage-Dataset'
repo_dir = os.path.join(DATA_ROOT, 'PlantVillage-Dataset')

if os.path.isdir(DATASET_DIR) and len(os.listdir(DATASET_DIR)) == 38:
    print('Dataset already downloaded - skipping.')
else:
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    REPO_URL, repo_dir], check=True)
    subprocess.run(['git', '-C', repo_dir, 'sparse-checkout', 'set', 'raw/color'], check=True)
    print(f'Download finished in {time.time() - t0:.0f} seconds')

n_images = sum(len(files) for _, _, files in os.walk(DATASET_DIR))
print('Class folders:', len(os.listdir(DATASET_DIR)))
print('Total images :', n_images)     # should print 54305

## Step 2 — List every image
We build a table where each row is one image and its class (label). The folder name **is** the label, e.g. `Tomato___Early_blight`.

In [ ]:
# ============================================================
# HELPER: turn folder names into readable names
# 'Tomato___Early_blight'  ->  'Tomato - Early blight'
# ============================================================
def pretty_name(folder_name):
    crop, disease = folder_name.split('___')
    crop = crop.replace('_', ' ').replace(',', '').strip()
    disease = disease.replace('_', ' ').strip()
    disease = disease[0].upper() + disease[1:]
    return f'{crop} - {disease}'

def crop_of(folder_name):
    return folder_name.split('___')[0].replace('_', ' ').replace(',', '').strip()

In [ ]:
# ============================================================
# BUILD A TABLE OF ALL IMAGES: one row per image -> (file path, class)
# ============================================================
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
CLASS_NAMES = sorted(d.name for d in Path(DATASET_DIR).iterdir() if d.is_dir())
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}

records = []
for c in CLASS_NAMES:
    for p in sorted((Path(DATASET_DIR) / c).iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            # store the path RELATIVE to the dataset folder (works in any session)
            records.append((f'{c}/{p.name}', c, CLASS_TO_ID[c]))

df = pd.DataFrame(records, columns=['filepath', 'label', 'label_id'])
print(f'{len(df)} images in {NUM_CLASSES} classes')

# Save the list of class names - the app needs it later to turn numbers back into names
with open(os.path.join(MODELS_DIR, 'class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f, indent=1)
df.head()

## Step 3 — How many images per class?
This checks **class balance**. If some classes have far fewer images, the model may ignore them, which is why we later use *class weights* and look at per-class metrics, not only accuracy.

In [ ]:
counts = df['label'].value_counts().reindex(CLASS_NAMES)

plt.figure(figsize=(12, 10))
order = counts.sort_values()
colors = ['#2e7d32' if 'healthy' in c else '#c62828' for c in order.index]
plt.barh([pretty_name(c) for c in order.index], order.values, color=colors)
plt.xlabel('Number of images')
plt.title('PlantVillage: images per class  (green = healthy, red = diseased)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'class_distribution.png'), dpi=150)
plt.show()

print(f'Largest class : {pretty_name(counts.idxmax())} ({counts.max()} images)')
print(f'Smallest class: {pretty_name(counts.idxmin())} ({counts.min()} images)')
print(f'Imbalance ratio (largest / smallest): {counts.max() / counts.min():.1f}x')

In [ ]:
# Images per crop, and how many classes each crop has
crop_table = (df.assign(crop=df['label'].map(crop_of),
                        healthy=df['label'].str.contains('healthy'))
                .groupby('crop')
                .agg(images=('filepath', 'count'),
                     classes=('label', 'nunique'),
                     healthy_images=('healthy', 'sum'))
                .sort_values('images', ascending=False))
crop_table

## Step 4 — What do the images look like?
One example photo from every class.

In [ ]:
n = NUM_CLASSES
cols = 6
rows = int(np.ceil(n / cols))
plt.figure(figsize=(cols * 2.6, rows * 2.9))
rng = np.random.default_rng(SEED)
from PIL import Image
for i, c in enumerate(CLASS_NAMES):
    files = df.loc[df['label'] == c, 'filepath'].values
    img = Image.open(os.path.join(DATASET_DIR, rng.choice(files)))
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(pretty_name(c).replace(' - ', '\n'), fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'sample_images.png'), dpi=120)
plt.show()

## Step 5 — Image properties
We check the size and colour format of a random sample of images.

In [ ]:
sample = df.sample(min(500, len(df)), random_state=SEED)
sizes, modes = [], []
for p in sample['filepath']:
    with Image.open(os.path.join(DATASET_DIR, p)) as im:
        sizes.append(im.size); modes.append(im.mode)
print('Image sizes found (width, height):', pd.Series(sizes).value_counts().to_dict())
print('Colour modes found:', pd.Series(modes).value_counts().to_dict())

## Step 6 — Train / Validation / Test split
- **Train (70%)** — the model learns from these.
- **Validation (15%)** — checked after every training round to tune settings and stop at the right time.
- **Test (15%)** — kept hidden until the very end and used once for the final, honest score.

In [ ]:
# ============================================================
# SPLIT INTO TRAIN (70%) / VALIDATION (15%) / TEST (15%)
# 'stratify' keeps the same class proportions in every part,
# so rare classes (e.g. Potato healthy, 152 images) appear in all three.
# The split is saved, so every notebook uses EXACTLY the same images.
# ============================================================
from sklearn.model_selection import train_test_split

split_files = {s: os.path.join(RESULTS_DIR, f'split_{s}.csv') for s in ['train', 'val', 'test']}

if all(os.path.exists(f) for f in split_files.values()):
    train_df, val_df, test_df = (pd.read_csv(split_files[s]) for s in ['train', 'val', 'test'])
    print('Loaded the saved split.')
else:
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label_id'],
                                         random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label_id'],
                                       random_state=SEED)
    for s, part in zip(['train', 'val', 'test'], [train_df, val_df, test_df]):
        part.reset_index(drop=True).to_csv(split_files[s], index=False)
    train_df, val_df, test_df = (pd.read_csv(split_files[s]) for s in ['train', 'val', 'test'])
    print('Created and saved a new split.')

print(f'Train: {len(train_df):6d} images')
print(f'Val  : {len(val_df):6d} images')
print(f'Test : {len(test_df):6d} images')

In [ ]:
# Check that every class appears in the same proportion in all three parts
dist = pd.DataFrame({
    'train': train_df['label'].value_counts(),
    'val':   val_df['label'].value_counts(),
    'test':  test_df['label'].value_counts(),
}).reindex(CLASS_NAMES)
dist.index = [pretty_name(c) for c in dist.index]
dist.to_csv(os.path.join(RESULTS_DIR, 'split_class_counts.csv'))

(dist / dist.sum()).plot(kind='bar', figsize=(16, 5), width=0.8)
plt.ylabel('Share of the split'); plt.title('Class proportions are the same in train, val and test (stratified split)')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'split_distribution.png'), dpi=130); plt.show()
dist.head(10)

## Step 7 — Preprocessing pipeline
We use TensorFlow's `tf.data` to load images in batches efficiently:
1. **Read** the file from disk
2. **Decode** the JPEG into a grid of numbers (height × width × 3 colour channels)
3. **Resize** to 224 × 224 (standard input size for CNNs such as MobileNetV2)
4. Group into **batches** of 32 and **prefetch** the next batch while the GPU works

In [ ]:
# ============================================================
# PREPROCESSING PIPELINE (tf.data)
# For every image:  read file -> decode JPEG -> resize to 224x224
# Normalisation (0-255 -> 0-1) is done INSIDE the model as its first layer,
# so the saved model can be given a raw photo directly (this keeps the app simple).
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])      # float32, values 0-255
    return img, label

def make_dataset(frame, shuffle=False):
    paths = [os.path.join(DATASET_DIR, p) for p in frame['filepath']]
    ds = tf.data.Dataset.from_tensor_slices((paths, frame['label_id'].values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)       # NOT shuffled -> predictions stay in the same order as test_df

images, labels = next(iter(train_ds))
print('One batch of images:', images.shape, images.dtype)
print('One batch of labels:', labels.shape)

## Step 8 — Normalisation
Pixel values go from 0 (black) to 255 (bright). Neural networks learn better with small numbers, so we divide by 255 to get values between 0 and 1. A `Rescaling(1/255)` layer does this as the **first layer of the model**.

In [ ]:
normalise = layers.Rescaling(1. / 255)
raw = images[:1]
print('Before normalisation: min =', float(tf.reduce_min(raw)), ' max =', float(tf.reduce_max(raw)))
norm = normalise(raw)
print('After  normalisation: min =', float(tf.reduce_min(norm)), ' max =', float(tf.reduce_max(norm)))

## Step 9 — Data augmentation
During training each image is randomly flipped, rotated, zoomed and contrast-changed, giving the model "new" variations of the same photo every round. This reduces **overfitting** (memorising the training photos instead of learning the disease patterns).

In [ ]:
# ============================================================
# DATA AUGMENTATION: random flips, rotations, zooms and contrast changes.
# Only active during TRAINING - it turns 1 photo into many slightly different ones,
# so the model learns "what a disease looks like", not "where the leaf sits in the photo".
# ============================================================
def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),     # up to +/- 54 degrees
        layers.RandomZoom(0.15),         # zoom in/out up to 15%
        layers.RandomContrast(0.15),
    ], name='data_augmentation')

In [ ]:
augment = make_augmentation()
img = images[0]
plt.figure(figsize=(14, 3.2))
plt.subplot(1, 7, 1); plt.imshow(img.numpy().astype('uint8')); plt.title('Original'); plt.axis('off')
for i in range(6):
    aug = augment(tf.expand_dims(img, 0), training=True)[0]
    plt.subplot(1, 7, i + 2); plt.imshow(np.clip(aug.numpy(), 0, 255).astype('uint8'))
    plt.title(f'Augmented {i+1}'); plt.axis('off')
plt.suptitle(pretty_name(CLASS_NAMES[int(labels[0])]))
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'augmentation_examples.png'), dpi=130); plt.show()

## ✅ Phase 1 summary
| Item | Result |
|---|---|
| Dataset | PlantVillage (Hughes & Salathé, 2015): real, expert-labelled leaf photos |
| Size | 54,305 images, 38 classes, 14 crops |
| Image format | 256 × 256 colour JPEG → resized to 224 × 224 |
| Split | 70% train / 15% validation / 15% test (stratified, seed 42) |
| Preprocessing | resize → normalise (÷255) → augmentation (flip, rotate, zoom, contrast) |
| Known challenge | class imbalance (152 to 5,507 images per class) → class weights + per-class metrics |

**Next:** `02_train_and_evaluate.ipynb` trains the CNN and MobileNetV2 and evaluates them.